# 🧠 Brain Tumor Classification Training

**Project**: Evidence-Aware Brain Tumour Classification System  
**Date**: November 9, 2026  
**GPU**: Tesla T4 (Free on Colab!)

---

## Steps:
1. ✅ Enable GPU (Runtime > Change runtime type > GPU)
2. ✅ Upload your code
3. ✅ Download dataset
4. ✅ Start training
5. ✅ Download results

## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not enabled! Enable it in Runtime > Change runtime type")

## Step 2: Upload Your Code

**Option A**: Upload ZIP file of your project  
**Option B**: Clone from GitHub (if you pushed it)

In [ ]:
# Option A: Upload project ZIP
from google.colab import files
import zipfile
import os

print("Please upload your Final_year_project.zip file...")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✓ Extraction complete!")

# Navigate to project
%cd Final_year_project

In [ ]:
# Option B: Clone from GitHub (if available)
!git clone https://github.com/YOUR_USERNAME/Final_year_project.git
%cd Final_year_project

## Step 3: Install Requirements

In [ ]:
# Install required packages
!pip install -q scikit-learn scikit-image matplotlib seaborn tqdm
print("✓ All packages installed!")

## Step 4: Download BRISC Dataset

**Option A**: Upload your downloaded dataset  
**Option B**: Download directly from Figshare

In [ ]:
# Option A: Upload dataset ZIP
from google.colab import files

print("Please upload brisc2025.zip...")
uploaded = files.upload()

# Extract
import zipfile
!mkdir -p data/brisc
with zipfile.ZipFile('brisc2025.zip', 'r') as zip_ref:
    zip_ref.extractall('data/brisc/')
print("✓ Dataset extracted!")

In [ ]:
# Option B: Download from Figshare (faster)
import os

print("Downloading BRISC dataset from Figshare...")
!mkdir -p data/brisc
!wget -q --show-progress https://figshare.com/ndownloader/files/51677885 -O brisc2025.zip
print("\nExtracting...")
!unzip -q brisc2025.zip -d data/brisc/
print("✓ Dataset ready!")

# Verify
!ls -lh data/brisc/brisc2025/classification_task/

## Step 5: Verify Setup

In [ ]:
# Quick verification
import os

checks = [
    ("train.py", os.path.exists("train.py")),
    ("src/models", os.path.exists("src/models")),
    ("src/data", os.path.exists("src/data")),
    ("Dataset", os.path.exists("data/brisc/brisc2025")),
]

print("Setup Verification:")
for name, exists in checks:
    status = "✓" if exists else "✗"
    print(f"  {status} {name}")

if all(exists for _, exists in checks):
    print("\n✅ All checks passed! Ready to train!")
else:
    print("\n⚠️ Some checks failed. Please review above.")

## Step 6: Quick Test (5 minutes)

Test that everything works with 2 epochs:

In [ ]:
# Quick test run
!python train.py \
  --epochs 2 \
  --batch-size 8 \
  --device cuda

## Step 7: Full Training (2-3 hours)

Choose your configuration:

In [ ]:
# Configuration 1: Fast Baseline (30 minutes)
!python train.py \
  --backbone resnet18 \
  --epochs 30 \
  --batch-size 64 \
  --lr 1e-3 \
  --device cuda

In [ ]:
# Configuration 2: Recommended (2-3 hours) ⭐
!python train.py \
  --backbone resnet50 \
  --epochs 50 \
  --batch-size 32 \
  --lr 1e-4 \
  --optimizer adam \
  --scheduler cosine \
  --augmentation medium \
  --early-stopping 10 \
  --device cuda

In [ ]:
# Configuration 3: Maximum Accuracy (6-8 hours)
!python train.py \
  --backbone resnet101 \
  --freeze-backbone \
  --unfreeze-after 20 \
  --epochs 100 \
  --batch-size 16 \
  --lr 5e-5 \
  --optimizer adamw \
  --augmentation heavy \
  --early-stopping 15 \
  --use-amp \
  --device cuda

## Step 8: View Results

In [ ]:
# Display training curves
from IPython.display import Image, display

print("Training Curves:")
display(Image('models/checkpoints/training_curves.png'))

print("\nConfusion Matrix:")
display(Image('models/checkpoints/confusion_matrix.png'))

In [ ]:
# Check training history
import json

with open('models/checkpoints/training_history.json', 'r') as f:
    history = json.load(f)

print("Training Summary:")
print(f"Final Train Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
print(f"Best Val Accuracy: {max(history['val_acc']):.2f}%")
print(f"Best Epoch: {history['val_acc'].index(max(history['val_acc'])) + 1}")

## Step 9: Download Results

In [ ]:
# Download trained model and results
from google.colab import files

print("Downloading files...")
files.download('models/checkpoints/best_model.pth')
files.download('models/checkpoints/training_curves.png')
files.download('models/checkpoints/confusion_matrix.png')
files.download('models/checkpoints/training_history.json')
print("✓ Download complete!")

## Step 10: Test Inference (Optional)

In [ ]:
# Load model and test on a single image
import torch
from PIL import Image
import sys
sys.path.insert(0, 'src')

from models.tumour_classifier import load_model
from data import Preprocessor

# Load trained model
model = load_model('models/checkpoints/best_model.pth', device='cuda')
model.eval()

# Load and preprocess image
preprocessor = Preprocessor(image_size=224)
image_path = 'data/brisc/brisc2025/classification_task/test/glioma/brisc2025_test_00001_gl_ax_t1.jpg'
image = Image.open(image_path).convert('RGB')
image_tensor = preprocessor.preprocess_image(image).unsqueeze(0).cuda()

# Predict
with torch.no_grad():
    probabilities, prediction = model.predict(image_tensor)

class_names = ['glioma', 'meningioma', 'pituitary']
print(f"\nPrediction: {class_names[prediction.item()]}")
print(f"Confidence: {probabilities[0, prediction].item():.2%}")
print(f"\nAll probabilities:")
for i, class_name in enumerate(class_names):
    print(f"  {class_name:12s}: {probabilities[0, i].item():.2%}")

# Display image
from IPython.display import display
display(image)

---

## 🎉 Training Complete!

**Next Steps**:
1. ✅ Download your trained model
2. ✅ Analyze results
3. ✅ Proceed to Increment 2: Grad-CAM Implementation

**Expected Results**:
- Validation Accuracy: 92-96%
- Per-class F1-Score: 0.92-0.96
- Training time: 2-3 hours

---

**Project**: Evidence-Aware Brain Tumour Classification System  
**Date**: November 9, 2026  
**Status**: Increment 1 Complete ✅